# 🎓 Train your LLM on a free GPU (Google Colab)

This notebook fine-tunes your model on a **free GPU**, then lets you **download the trained adapter** to deploy on Railway.

**The flow:** train here (fast, free GPU) → download `adapter.zip` → put it in your project → push to GitHub → Railway redeploys with your fine-tuned model.

➡️ **First: turn on the GPU.** Menu **Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save.**

## Step 1 — Confirm the GPU is on

In [ ]:
!nvidia-smi
import torch
print('GPU available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU! Go to Runtime > Change runtime type > T4 GPU, then re-run.'

## Step 2 — Get your project code
Clones your GitHub repo so we use the exact same `train.py`, `config.py`, and data.

In [ ]:
%cd /content
![ -d LLM-Model ] || git clone https://github.com/zaheer23831-dev/LLM-Model.git
%cd /content/LLM-Model
!git pull -q
print('\nProject ready.')

In [ ]:
# Install the ML libraries (Colab already has a GPU build of torch).
!pip install -q transformers peft accelerate huggingface_hub
print('Dependencies installed.')

## Step 3 — Your training data
By default this uses the sample `data/train.jsonl` from the repo.

**To train on YOUR data:** run the cell below and upload your own `train.jsonl` (one JSON object per line: `{"instruction": ..., "input": ..., "output": ...}`). Skip the cell to use the sample.

In [ ]:
from google.colab import files
import shutil
print('Upload your train.jsonl (or skip this cell to use the repo sample):')
uploaded = files.upload()
for name in uploaded:
    if name.endswith('.jsonl'):
        shutil.move(name, 'data/train.jsonl')
        print(f'\nSaved {name} -> data/train.jsonl')

# Show how many examples we'll train on
with open('data/train.jsonl', encoding='utf-8') as f:
    n = sum(1 for line in f if line.strip())
print(f'Training on {n} examples.')

## Step 4 — (Optional) Settings
The GPU can handle a bigger model and more epochs than your laptop. **Uncomment** any line below to override the defaults, then run the cell.

In [ ]:
# %env BASE_MODEL_ID=Qwen/Qwen2.5-1.5B-Instruct   # bigger, smarter (GPU handles it)
# %env EPOCHS=5                                    # more passes over your data
# %env LR=2e-4
# %env MAX_SEQ_LEN=512
print('Settings ready (defaults unless you uncommented overrides above).')

## Step 5 — Train 🚀
Runs your `train.py`, which auto-detects the GPU and uses mixed precision. This is the step that's *minutes on GPU* vs *hours on CPU*.

In [ ]:
!python train.py

## Step 6 — Quick test
Chat with your freshly fine-tuned model to sanity-check it.

In [ ]:
!python -c "import inference; print(inference.generate('Who are you?', max_new_tokens=60))"

## Step 7 — Download your adapter
Downloads `adapter.zip` (your fine-tuned LoRA, ~35 MB) to your computer.

In [ ]:
import shutil
from google.colab import files
shutil.rmtree('models/adapter/_checkpoints', ignore_errors=True)  # not needed
shutil.make_archive('adapter', 'zip', 'models/adapter')
print('Zipped. Downloading...')
files.download('adapter.zip')

## Step 8 — Deploy to Railway
On your laptop, inside your project folder:

1. **Unzip** `adapter.zip` into `models/adapter/` (replace the old contents).
2. **Commit & push** — the repo is set up to include `models/adapter/`:
   ```bash
   git add models/adapter
   git commit -m "Add fine-tuned adapter from Colab"
   git push
   ```
3. **Redeploy** so Railway picks it up:
   ```bash
   railway up --detach
   ```

Your live app at **llm-model-production.up.railway.app** now serves your fine-tuned model. 🎉

_(Because the adapter is baked into the image, it persists across redeploys — no Volume needed.)_